# CNN : variations d'architecture

**Objectifs.**
- Retrouver la batch normalization en contexte convolutif
  (`nn.BatchNorm2d`).
- Comprendre et coder _from scratch_ un bloc résiduel (ResNet) : pourquoi ça aide à
  entraîner des réseaux plus profonds.
- Comparer empiriquement un mini-VGG et une version à profondeur comparable munie
  de connexions résiduelles.

Vous allez utiliser le dataset Fashion-MNIST qui est plus difficile que celui de la séance précédente, et on vous fournit une architecture de référence (mini-VGG).


In [ ]:
# !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from training_toolbox import Trainer, accuracy, count_trainable_parameters

torch.manual_seed(0)


## Partie 0 — Données


In [ ]:
CLASSES = [
    "T-shirt/top", "Pantalon", "Pull", "Robe", "Manteau",
    "Sandale", "Chemise", "Basket", "Sac", "Bottine",
]

transform = transforms.ToTensor()

full_train = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

n_val = 5000
n_train = len(full_train) - n_val
train_set, val_set = random_split(full_train, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)


In [ ]:
x_batch, y_batch = next(iter(train_loader))
print("Shape d'un batch d'images :", x_batch.shape)  # (batch, 1, 28, 28)
print("Shape des labels :", y_batch.shape)

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, img, label in zip(axes.ravel(), x_batch, y_batch):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(CLASSES[label.item()], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Partie 1 — l'architecture mini-VGG

On utilise comme référence l'architecture ci-dessous :


In [ ]:
class MiniVGG(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## Partie 2 — Batch normalization en contexte convolutif

La batch normalization normalise les activations (moyenne 0, variance 1
sur le batch), puis applique une transformation affine apprise. En contexte convolutif,
`nn.BatchNorm2d(C)` normalise **par canal** `C`, sur l'ensemble des positions spatiales et du
batch — un seul couple (moyenne, variance) par canal, pas par pixel.

**Question 2.1.** Codez une variante `MiniVGGBatchNorm` ci-dessous pour arriver à une architecture similaire au mini-VGG de la
Partie 1, avec un `nn.BatchNorm2d` ajouté après chaque convolution et avant chaque `ReLU`.


In [ ]:
class MiniVGGBatchNorm(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass

**Question.** `nn.BatchNorm2d` se comporte différemment en mode `train()` et en mode
`eval()` (tout comme `nn.Dropout`). Quelle statistique utilise-t-il dans chaque cas, et
pourquoi cette différence est-elle nécessaire ?


_Votre réponse ici._

## Partie 3 — Un bloc résiduel, from scratch

**Le problème de dégradation.** Au-delà d'une certaine profondeur, empiler plus de couches
convolutives *dégrade* les performances — y compris sur le train set, ce qui n'est pas un
problème de sur-apprentissage mais un problème d'**optimisation** : le réseau a du mal à
apprendre, même la fonction identité, à travers de nombreuses couches non linéaires
empilées.

**Question 3.1.** Coder un bloc résiduel `ResidualBlock` (inspiré de ResNet) composé de deux convolutions 3×3 (quel padding ?), avec batch normalization et ReLU. 
La connection résiduelle (_skip connection_) est déjà implémentée. Est-elle exactement équivelente à la formule du cours ?


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.relu = nn.ReLU()

        # TODO : chemin principal (2 convolutions 3x3 + BatchNorm2d, cf. l'énoncé)

        # Raccourci : déjà fourni, rien à faire ici.
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        # TODO : chemin principal, puis ajout du raccourci et ReLU final
        pass


# Vérification rapide : un bloc qui change de résolution et de nombre de canaux
block = ResidualBlock(in_channels=32, out_channels=64, stride=2)
dummy = torch.randn(4, 32, 14, 14)
print("Entrée :", dummy.shape, "-> Sortie :", block(dummy).shape)  # attendu : (4, 64, 7, 7)

## Partie 4 — Un mini-ResNet, à profondeur comparable au mini-VGG

**Question 4.1.** Le mini-VGG a 4 couches convolutives (2 blocs de 2 convolutions). Construisez un réseau
avec le même nombre de convolutions, organisées en 2 blocs résiduels (chaque
`ResidualBlock` contient 2 convolutions).


In [ ]:
class MiniResNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass


model_resnet = MiniResNet()
with torch.no_grad():
    out = model_resnet(torch.randn(2, 1, 28, 28))
print("Sortie MiniResNet :", out.shape)
print(f"Paramètres MiniResNet   : {count_trainable_parameters(model_resnet):,}")

## Partie 5 — Comparaison empirique

**Question 5.1.** Entraînez les trois variantes (mini-VGG simple, mini-VGG + BatchNorm, MiniResNet) sur les
mêmes données, avec les mêmes hyperparamètres, et comparez les courbes d'apprentissage.


In [ ]:
def train_and_report(model, name, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
    print(f"--- {name} ({count_trainable_parameters(model):,} paramètres) ---")
    history = trainer.fit(train_loader, val_loader, epochs=epochs, verbose=True)
    return history


histories = {}
histories["mini-VGG"] = train_and_report(MiniVGG(), "mini-VGG")
histories["mini-VGG + BatchNorm"] = train_and_report(MiniVGGBatchNorm(), "mini-VGG + BatchNorm")
histories["MiniResNet"] = train_and_report(MiniResNet(), "MiniResNet")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    axes[0].plot(h["val_loss"], label=name)
    axes[1].plot(h["val_acc"], label=name)
axes[0].set_title("Validation loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()


**Questions de compréhension.**

- Classez les trois modèles par accuracy de validation, et par vitesse de convergence
  (nombre d'epochs pour atteindre une accuracy donnée). Les observations correspondent-elles
  à ce que vous attendiez ?
- Les architectures utilisées ici restent peu profondes (4 convolutions) : à cette
  profondeur, le "problème de dégradation" évoqué en cours (des réseaux *très* profonds, type
  ResNet à 50 ou 100+ couches, qui n'arrivent plus à apprendre l'identité) n'est pas forcément
  très marqué. Qu'observez-vous malgré tout comme différence de comportement entre le
  mini-VGG et le MiniResNet ?
- BatchNorm et skip connections sont deux mécanismes différents mais souvent combinés en
  pratique. Sur la base de vos courbes, essayez de séparer ce qui relève de l'un et de
  l'autre (indice : comparez d'abord mini-VGG vs mini-VGG+BN, puis mini-VGG+BN vs
  MiniResNet, qui contient BN *et* les skip connections).

## Pour aller plus loin (optionnel)

Empiler 2 ou 3 `ResidualBlock` supplémentaires (profondeur plus réaliste) et observer si
  l'écart avec un mini-VGG "empilé" de la même façon (sans skip connections) se creuse.
